# 实践 6.1  实测表达与遗传预测表达为什么不同

《统计遗传与智能育种学》第六章 · 电子资源 `06_twas_grex.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：06_twas_grex.ipynb。实践模拟一组cis-SNP、一个由遗传组分和环境共同决定的表达量，以及同时受遗传组分与共享环境影响的数量性状。共享环境使实测表达—性状相关受到非遗传共同原因影响；随后用带折内标准化的岭回归和5折交叉验证（cross-validation, CV）预测遗传调控表达（GReX），避免在同一数据上训练并评价表达预测模型。汇总TWAS示例另将样本分成独立的表达参考子集和GWAS目标子集：前者估计表达权重，后者计算近似GWAS Z与LD矩阵，并把表达权重换算到与目标子集Z、LD一致的基因型尺度。

关键输出：固定SEED=601时，实测表达—性状相关约0.63，真实遗传表达—性状相关约0.42，交叉验证GReX—表达约0.47、GReX—性状约0.35；分离表达参考子集与GWAS目标子集后的简化汇总TWAS Z约6.55；本次置换后GReX—性状相关约0。  
解释重点：实测表达相关较强不等于遗传因果证据较强；简化Z统计量用于解释表达权重、GWAS Z和LD之间的组合关系，不替代FUSION或S-PrediXcan的正式实现、参考面板设计和统计校准。

## 输入、完整代码、实际输出与解释

输入与可复现条件：cis-SNP、遗传表达组分、共享环境与数量性状；SEED=601；5折交叉验证中标准化只在训练折拟合。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=601
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold,cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

n,m=500,40; p=rng.uniform(.08,.5,m); G=rng.binomial(2,p,(n,m)).astype(float); Gc=G-G.mean(0); wtrue=np.zeros(m); wtrue[:8]=np.array([.50,-.45,.40,.35,-.40,.30,-.35,.45]); genetic_expr=Gc@wtrue
# shared_env同时影响表达和性状，用来演示实测表达相关可被非遗传共同原因增强
shared_env=rng.normal(0,1,n); expr=genetic_expr+.9*shared_env+rng.normal(0,.6,n); y=.8*genetic_expr+.8*shared_env+rng.normal(0,1.0,n)
print('measured-expression r',round(np.corrcoef(expr,y)[0,1],3),'true-genetic-expression r',round(np.corrcoef(genetic_expr,y)[0,1],3))

# GReX必须在训练折中学习表达权重，不能先用全数据拟合再宣称交叉验证
model=make_pipeline(StandardScaler(),RidgeCV(alphas=np.logspace(-2,3,20))); cv=KFold(5,shuffle=True,random_state=SEED); grex=cross_val_predict(model,G,expr,cv=cv); print('CV GReX-expression r',round(np.corrcoef(grex,expr)[0,1],3),'CV GReX-trait r',round(np.corrcoef(grex,y)[0,1],3))
# 汇总TWAS示意：独立参考子集学习表达权重，目标子集只提供GWAS Z与LD。
ref_idx=np.arange(0,250); gwas_idx=np.arange(250,n)
sc_ref=StandardScaler().fit(G[ref_idx]); X_ref=sc_ref.transform(G[ref_idx]); fit=RidgeCV(alphas=np.logspace(-2,3,20)).fit(X_ref,expr[ref_idx]); w_allele=fit.coef_/sc_ref.scale_
sc_gwas=StandardScaler().fit(G[gwas_idx]); X_gwas=sc_gwas.transform(G[gwas_idx]); w_gwas=w_allele*sc_gwas.scale_; y_gwas=(y[gwas_idx]-y[gwas_idx].mean())/y[gwas_idx].std(); n_gwas=len(gwas_idx)
z=np.array([(X_gwas[:,j]@y_gwas)/np.sqrt(n_gwas) for j in range(m)]); LD=np.corrcoef(X_gwas,rowvar=False); denom=float(np.sqrt(max(w_gwas@LD@w_gwas,1e-12))); z_twas=float(w_gwas@z/denom); print('simplified summary TWAS Z',round(z_twas,3),'denominator',round(denom,3))
perm_y=rng.permutation(y); print('permuted measured-expression r',round(np.corrcoef(expr,perm_y)[0,1],3),'permuted GReX-trait r',round(np.corrcoef(grex,perm_y)[0,1],3))


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 601
measured-expression r 0.632 true-genetic-expression r 0.415
CV GReX-expression r 0.468 CV GReX-trait r 0.351
simplified summary TWAS Z 6.547 denominator 0.424
permuted measured-expression r -0.093 permuted GReX-trait r -0.0


结果解释：实测表达—性状相关0.632高于真实遗传表达—性状相关0.415，是因为模拟共享环境同时影响两者；CV GReX—表达相关0.468、GReX—性状相关0.351。汇总示例中表达权重由前250个样本学习，GWAS Z与LD由后250个样本计算，避免用同一批个体同时承担表达参考集和GWAS目标集。简化汇总TWAS Z=6.547仅用于展示权重、GWAS Z与LD的组合关系，不替代FUSION或S-PrediXcan，也不能单独证明表达介导因果。

可修改练习：把shared_env对性状的系数0.8改为0，再比较实测表达、真实遗传表达和CV GReX三类相关。